In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
 
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
 
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")
 
spark = SparkSession.builder \
    .appName("Revenue Forecasting XGBoost") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0") \
    .getOrCreate()
 
spark.sparkContext.setLogLevel("ERROR")
 
BASE_PATH = "data"   # chỉnh lại nếu cần
 
print("Spark ready:", spark.version)

Spark ready: 3.5.8


In [ ]:
BASE_PATH = r"D:\Hoc_hanh\BigData\Lakehouse_instacart\data"
daily_sales   = spark.read.format("delta").load(f"{BASE_PATH}/gold/daily_sales")
traffic_kpi   = spark.read.format("delta").load(f"{BASE_PATH}/gold/traffic_kpi")
payment_ana   = spark.read.format("delta").load(f"{BASE_PATH}/gold/payment_analytics")
inventory_kpi = spark.read.format("delta").load(f"{BASE_PATH}/gold/inventory_kpi")
returns_kpi   = spark.read.format("delta").load(f"{BASE_PATH}/gold/returns_kpi")
review_ana    = spark.read.format("delta").load(f"{BASE_PATH}/gold/review_analytics")
 
print("Gold tables loaded.")
daily_sales.printSchema()


Gold tables loaded.
root
 |-- event_date: date (nullable = true)
 |-- gross_revenue: double (nullable = true)
 |-- shipping_revenue: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_products: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- gold_layer: string (nullable = true)



In [35]:
daily_sales.show(5)
print(daily_sales.count())

+-------------+-------------+----------------+-------------+------------+--------------+---------------+----------+
|   event_date|gross_revenue|shipping_revenue|total_revenue|total_orders|total_products|avg_order_value|gold_layer|
+-------------+-------------+----------------+-------------+------------+--------------+---------------+----------+
| +67058-01-01|         NULL|            NULL|         NULL|           1|             1|           NULL|      gold|
|+205722-01-01|         NULL|            NULL|         NULL|           1|             1|           NULL|      gold|
|   9606-01-01|         NULL|            NULL|         NULL|           1|             1|           NULL|      gold|
| +71729-01-01|         NULL|            NULL|         NULL|           1|             1|           NULL|      gold|
| +88455-01-01|         NULL|            NULL|         NULL|           1|             1|           NULL|      gold|
+-------------+-------------+----------------+-------------+------------

In [27]:
from pyspark.sql.functions import *

def normalize_event_date(df):
    return df.withColumn(
        "event_date",
        coalesce(
            to_date(col("event_date"), "yyyy-MM-dd"),
            to_date(col("event_date"), "yyyyMMdd"),
            to_date(col("event_date"), "dd/MM/yyyy")
        )
    )

In [28]:
traffic_kpi   = normalize_event_date(traffic_kpi)
payment_ana   = normalize_event_date(payment_ana)
inventory_kpi = normalize_event_date(inventory_kpi)
returns_kpi   = normalize_event_date(returns_kpi)
review_ana    = normalize_event_date(review_ana)
daily_sales   = normalize_event_date(daily_sales)

In [29]:
from pyspark.sql.functions import year

daily_sales = daily_sales.filter(
    (col("event_date").isNotNull()) &
    (year("event_date") < 2030) &
    (year("event_date") > 2000)
)

In [30]:
# CELL 3 — Aggregate Sub-tables to Daily Grain
# --- Traffic: tổng theo ngày ---
traffic_daily = traffic_kpi.groupBy("event_date").agg(
    sum("total_sessions").alias("total_sessions"),
    sum("unique_visitors").alias("unique_visitors"),
    sum("page_views").alias("page_views"),
    avg("avg_pages_per_session").alias("avg_pages_per_session")
)
 
# --- Payments: tổng theo ngày (pivot payment_method nếu muốn) ---
payment_daily = payment_ana.groupBy("event_date").agg(
    sum("total_transactions").alias("total_transactions"),
    sum("payment_total").alias("payment_total"),
    avg("avg_payment").alias("avg_payment"),
    sum("fraud_transactions").alias("fraud_transactions")
)

# --- Inventory ---
inventory_daily = inventory_kpi.groupBy("event_date").agg(
    sum("units_sold").alias("inv_units_sold"),
    avg("avg_stock").alias("avg_stock")
)
# --- Returns ---
returns_daily = returns_kpi.groupBy("event_date").agg(
    sum("total_returns").alias("total_returns"),
    sum("returned_orders").alias("returned_orders")
)
 
# --- Reviews ---
reviews_daily = review_ana.groupBy("event_date").agg(
    sum("total_reviews").alias("total_reviews"),
    avg("avg_rating").alias("avg_rating")
    )
 
print("Sub-aggregations done.")

Sub-aggregations done.


In [31]:
date_df = daily_sales.select("event_date").distinct()

In [32]:
# CELL 4 — Join tất cả thành Feature Table
feature_df = date_df \
    .join(daily_sales,     "event_date", "left") \
    .join(traffic_daily,   "event_date", "left") \
    .join(payment_daily,   "event_date", "left") \
    .join(inventory_daily, "event_date", "left") \
    .join(returns_daily,   "event_date", "left") \
    .join(reviews_daily,   "event_date", "left")
feature_df.show(5)
print("Rows:", feature_df.count())

+----------+-------------+----------------+-------------+------------+--------------+---------------+----------+--------------+---------------+----------+---------------------+------------------+-------------+-----------+------------------+--------------+---------+-------------+---------------+-------------+----------+
|event_date|gross_revenue|shipping_revenue|total_revenue|total_orders|total_products|avg_order_value|gold_layer|total_sessions|unique_visitors|page_views|avg_pages_per_session|total_transactions|payment_total|avg_payment|fraud_transactions|inv_units_sold|avg_stock|total_returns|returned_orders|total_reviews|avg_rating|
+----------+-------------+----------------+-------------+------------+--------------+---------------+----------+--------------+---------------+----------+---------------------+------------------+-------------+-----------+------------------+--------------+---------+-------------+---------------+-------------+----------+
+----------+-------------+-----------

In [37]:
# CELL 5 — Convert to Pandas & Basic Cleaning
pdf = feature_df.toPandas()
 
pdf["event_date"] = pd.to_datetime(pdf["event_date"])
pdf = pdf.sort_values("event_date").reset_index(drop=True)
 
# Fill NaN bằng 0 (ngày không có returns / reviews …)
num_cols = pdf.select_dtypes(include=[np.number]).columns
pdf[num_cols] = pdf[num_cols].fillna(0)
 
TARGET = "total_revenue"   # cột mục tiêu từ daily_sales
 
print(pdf.shape)
pdf.head(3)

ValueError: year 10018 is out of range

In [ ]:
# CELL 6 — Feature Engineering
df = pdf.copy()
 
# --- Thời gian ---
df["year"]          = df["event_date"].dt.year
df["month"]         = df["event_date"].dt.month
df["day"]           = df["event_date"].dt.day
df["day_of_week"]   = df["event_date"].dt.dayofweek   # 0=Mon
df["day_of_year"]   = df["event_date"].dt.dayofyear
df["week_of_year"]  = df["event_date"].dt.isocalendar().week.astype(int)
df["is_weekend"]    = df["day_of_week"].isin([5, 6]).astype(int)
df["quarter"]       = df["event_date"].dt.quarter
 
# --- Cyclical encoding cho tháng & ngày ---
df["month_sin"]     = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]     = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"]       = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]       = np.cos(2 * np.pi * df["day_of_week"] / 7)
 
# --- Lag features (target) ---
for lag in [1, 2, 3, 7, 14, 28]:
    df[f"revenue_lag_{lag}"] = df[TARGET].shift(lag)
 
# --- Rolling statistics ---
for window in [7, 14, 28]:
    df[f"revenue_roll_mean_{window}"] = (
        df[TARGET].shift(1).rolling(window).mean()
    )
    df[f"revenue_roll_std_{window}"]  = (
        df[TARGET].shift(1).rolling(window).std()
    )
    df[f"revenue_roll_max_{window}"]  = (
        df[TARGET].shift(1).rolling(window).max()
    )
 
# --- Lag traffic ---
for lag in [1, 7]:
    df[f"sessions_lag_{lag}"]        = df["total_sessions"].shift(lag)
    df[f"unique_visitors_lag_{lag}"] = df["unique_visitors"].shift(lag)
 
# --- Derived ratios ---
df["conversion_rate"] = np.where(
    df["total_sessions"] > 0,
    df["total_orders"] / df["total_sessions"], 0
)
df["return_rate"] = np.where(
    df["total_orders"] > 0,
    df["returned_orders"] / df["total_orders"], 0
)
df["avg_revenue_per_order"] = np.where(
    df["total_orders"] > 0,
    df[TARGET] / df["total_orders"], 0
)
df["revenue_per_visitor"] = np.where(
    df["unique_visitors"] > 0,
    df[TARGET] / df["unique_visitors"], 0
)
 
# Drop rows với NaN do lag/rolling
df = df.dropna().reset_index(drop=True)
 
print("After feature engineering:", df.shape)
df.head(3)
 

In [ ]:
# CELL 7 — Train / Validation / Test Split (time-based)
n = len(df)
train_end  = int(n * 0.70)
val_end    = int(n * 0.85)
 
FEATURE_COLS = [c for c in df.columns
                if c not in [TARGET, "event_date", "gold_layer"]]
 
X = df[FEATURE_COLS]
y = df[TARGET]
 
X_train, y_train = X.iloc[:train_end],  y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],    y.iloc[val_end:]
 
print(f"Train : {len(X_train)} rows  "
      f"({df['event_date'].iloc[0].date()} → "
      f"{df['event_date'].iloc[train_end-1].date()})")
print(f"Val   : {len(X_val)} rows  "
      f"({df['event_date'].iloc[train_end].date()} → "
      f"{df['event_date'].iloc[val_end-1].date()})")
print(f"Test  : {len(X_test)} rows  "
      f"({df['event_date'].iloc[val_end].date()} → "
      f"{df['event_date'].iloc[-1].date()})")

In [ ]:
# CELL 8 — Train XGBoost
xgb_model = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_weight  = 3,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,       # L1
    reg_lambda        = 1.0,       # L2
    gamma             = 0.0,
    random_state      = 42,
    eval_metric       = "rmse",
    early_stopping_rounds = 30,
    n_jobs            = -1
)
 
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)
 
best_iter = xgb_model.best_iteration
print(f"\nBest iteration: {best_iter}")

In [ ]:
# CELL 9 — Evaluate on Val & Test
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"  RMSE : {rmse:>12,.2f}")
    print(f"  MAE  : {mae:>12,.2f}")
    print(f"  R²   : {r2:>12.4f}")
    print(f"  MAPE : {mape:>11.2f}%")
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}
 
val_pred  = xgb_model.predict(X_val)
test_pred = xgb_model.predict(X_test)
 
val_metrics  = evaluate("VALIDATION SET", y_val.values,  val_pred)
test_metrics = evaluate("TEST SET",       y_test.values, test_pred)

In [ ]:
fi = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)
 
top_n = 20
fig, ax = plt.subplots(figsize=(10, 7))
fi.head(top_n).plot(kind="barh", ax=ax, color="#2196F3")
ax.invert_yaxis()
ax.set_title(f"Top {top_n} Feature Importances (XGBoost)", fontsize=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
 
# Validation
dates_val  = df["event_date"].iloc[train_end:val_end]
axes[0].plot(dates_val, y_val.values,  label="Actual",    color="#1565C0", lw=1.5)
axes[0].plot(dates_val, val_pred,      label="Predicted", color="#F44336", lw=1.5, linestyle="--")
axes[0].set_title("Validation Set — Actual vs Predicted", fontsize=13)
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
 
# Test
dates_test = df["event_date"].iloc[val_end:]
axes[1].plot(dates_test, y_test.values, label="Actual",    color="#1565C0", lw=1.5)
axes[1].plot(dates_test, test_pred,     label="Predicted", color="#FF9800", lw=1.5, linestyle="--")
axes[1].set_title("Test Set — Actual vs Predicted", fontsize=13)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
 
plt.tight_layout()
plt.show()

In [ ]:
test_residuals = y_test.values - test_pred
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
axes[0].scatter(test_pred, test_residuals, alpha=0.4, color="#9C27B0", s=20)
axes[0].axhline(0, color="black", lw=1.2, linestyle="--")
axes[0].set_xlabel("Predicted Revenue"); axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Predicted")
axes[0].grid(alpha=0.3)
 
axes[1].hist(test_residuals, bins=30, color="#9C27B0", alpha=0.7, edgecolor="white")
axes[1].axvline(0, color="black", lw=1.2, linestyle="--")
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")
axes[1].grid(alpha=0.3)
 
plt.suptitle("Residual Analysis — Test Set", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
FORECAST_DAYS = 30
 
last_row   = df.iloc[-1].copy()
last_date  = df["event_date"].iloc[-1]
history    = df[TARGET].values.tolist()   # để tính lag/rolling
 
forecast_records = []
 
for i in range(1, FORECAST_DAYS + 1):
    future_date = last_date + pd.Timedelta(days=i)
 
    row = {}
 
    # --- Temporal ---
    row["year"]         = future_date.year
    row["month"]        = future_date.month
    row["day"]          = future_date.day
    row["day_of_week"]  = future_date.dayofweek
    row["day_of_year"]  = future_date.timetuple().tm_yday
    row["week_of_year"] = future_date.isocalendar()[1]
    row["is_weekend"]   = int(future_date.dayofweek in [5, 6])
    row["quarter"]      = (future_date.month - 1) // 3 + 1
    row["month_sin"]    = np.sin(2 * np.pi * row["month"] / 12)
    row["month_cos"]    = np.cos(2 * np.pi * row["month"] / 12)
    row["dow_sin"]      = np.sin(2 * np.pi * row["day_of_week"] / 7)
    row["dow_cos"]      = np.cos(2 * np.pi * row["day_of_week"] / 7)
 
    # --- Lag revenue ---
    for lag in [1, 2, 3, 7, 14, 28]:
        row[f"revenue_lag_{lag}"] = history[-lag] if len(history) >= lag else 0
 
    # --- Rolling ---
    for w in [7, 14, 28]:
        chunk = history[-w:] if len(history) >= w else history
        row[f"revenue_roll_mean_{w}"] = np.mean(chunk)
        row[f"revenue_roll_std_{w}"]  = np.std(chunk) if len(chunk) > 1 else 0
        row[f"revenue_roll_max_{w}"]  = np.max(chunk)
 
    # --- Lag traffic (dùng median của 7 ngày gần nhất để estimate) ---
    for lag in [1, 7]:
        row[f"sessions_lag_{lag}"]        = df["total_sessions"].iloc[-lag]
        row[f"unique_visitors_lag_{lag}"] = df["unique_visitors"].iloc[-lag]
 
    # --- Các feature số liệu còn lại: dùng rolling mean 7 ngày ---
    for col in FEATURE_COLS:
        if col not in row:
            row[col] = df[col].iloc[-7:].mean()
 
    # --- Predict ---
    X_future = pd.DataFrame([row])[FEATURE_COLS]
    pred_rev  = float(xgb_model.predict(X_future)[0])
    pred_rev  = max(pred_rev, 0)
 
    forecast_records.append({
        "date":             future_date,
        "predicted_revenue": pred_rev
    })
    history.append(pred_rev)  # rolling update
 
forecast_df = pd.DataFrame(forecast_records)
 

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(14, 5))
history_plot = df[["event_date", TARGET]].tail(60)
ax.plot(history_plot["event_date"], history_plot[TARGET],
        label="Historical (last 60d)", color="#1565C0", lw=1.5)
ax.plot(forecast_df["date"], forecast_df["predicted_revenue"],
        label=f"Forecast (+{FORECAST_DAYS}d)",
        color="#E91E63", lw=2, linestyle="--", marker="o", markersize=4)
ax.axvline(last_date, color="gray", linestyle=":", lw=1.2, label="Forecast start")
ax.set_title(f"Revenue Forecast — Next {FORECAST_DAYS} Days", fontsize=14)
ax.legend(); ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 

In [ ]:
import joblib, json
 
joblib.dump(xgb_model, r"D:\Hoc_hanh\BigData\Lakehouse_instacart\models\xgb_revenue_model.pkl")
 
metrics_summary = {
    "best_iteration":  int(best_iter),
    "feature_count":   len(FEATURE_COLS),
    "train_rows":      len(X_train),
    "val_rows":        len(X_val),
    "test_rows":       len(X_test),
    "validation":      {k: round(v, 4) for k, v in val_metrics.items()},
    "test":            {k: round(v, 4) for k, v in test_metrics.items()}
}
 
with open("model_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
 
print("\n=== FINAL SUMMARY ===")
print(json.dumps(metrics_summary, indent=2))
print("\nModel saved: xgb_revenue_model.pkl")
print("Metrics saved: model_metrics.json")